# 教育シミュレーション時間経過 Notebook

最終更新: 2026-08-07 12:20:32 JST

このNotebookは、複数生徒AI・伝達AI・授業設計AIをつないで、授業シミュレーションを時間経過で回すための実行環境です。

目的は、認知モデルで生徒AIの正誤を制御し、LLMを任意で発話・伝達に入れながら、どこで問題が起きるか、どの評価指標が研究として使えそうかを確認することです。LLMを使う場合は、小規模LLM実行セルから確認します。


## このNotebookで見るもの

```text
教師beliefから授業設計
  -> 認知モデルが正誤・解答方針を決定
  -> 生徒AIが反応
  -> Observable Eventを生成
  -> 伝達AIがクラスを要約
  -> 教師beliefを更新
  -> 次サイクルの授業設計へ反映
```

標準ではLLMなしで高速に回します。正誤制御は `COGNITIVE_MODEL = "bkt_irt"` を使います。Colab GPUでLLMを使う場合だけ、設定セルのフラグをTrueにしてください。


In [ ]:
# Colab setup: clone/update repository, install dependencies, prepare imports

import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_ROOT = Path("/content/student-ai")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not PROJECT_ROOT.exists():
        !git clone {REPO_URL} /content/student-ai
    os.chdir(PROJECT_ROOT)
    !git fetch origin main
    !git reset --hard origin/main
    !git log -1 --oneline
    !pip install -q -r requirements.txt
else:
    PROJECT_ROOT = Path.cwd()
    os.chdir(PROJECT_ROOT)
    print("Local environment detected. Git update and pip install are skipped.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Colabではgit更新後も古いsrcモジュールが残ることがあるため、再importできるように消します。
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

for path in ["data/assessments", "data/logs/simulation_timeline", "data/teacher_beliefs/simulation_timeline", "data/logs/simulation_timeline_llm", "data/teacher_beliefs/simulation_timeline_llm"]:
    Path(path).mkdir(parents=True, exist_ok=True)

print("project_root:", PROJECT_ROOT)
print("in_colab:", IN_COLAB)
print("src import path ready:", (PROJECT_ROOT / "src").exists())
print("src modules reloaded:", True)


## 1. 実験設定

LLMを使う場合:

- ColabのランタイムをGPUにする
- `USE_LLM_STUDENT` または `USE_LLM_COMMUNICATION` を `True` にする
- 初回ロードは数分以上かかることがあります

研究の切り分けとしては、まず `USE_LLM_COMMUNICATION=True` から試すのがおすすめです。

通常実験はmockのままにして、LLMは後半の小規模セルで確認します。


In [ ]:
CLASS_ID = "class_10_mixed"
CLASS_SIZE = 5
CYCLES = 3
TOTAL_MINUTES = 30

USE_LLM_STUDENT = False
USE_LLM_COMMUNICATION = False
MODEL_ID = "Qwen/Qwen3-4B"
LOAD_IN_4BIT = True
COGNITIVE_MODEL = "bkt_irt"

# Trueにすると授業後に生徒状態を更新します。最初はFalseで、観察と推定の安定性を見ます。
UPDATE_STUDENT_KNOWLEDGE = False

config = {
    "CLASS_ID": CLASS_ID,
    "CLASS_SIZE": CLASS_SIZE,
    "CYCLES": CYCLES,
    "USE_LLM_STUDENT": USE_LLM_STUDENT,
    "USE_LLM_COMMUNICATION": USE_LLM_COMMUNICATION,
    "MODEL_ID": MODEL_ID,
    "LOAD_IN_4BIT": LOAD_IN_4BIT,
    "COGNITIVE_MODEL": COGNITIVE_MODEL,
    "UPDATE_STUDENT_KNOWLEDGE": UPDATE_STUDENT_KNOWLEDGE,
}
config


In [ ]:
# Optional: GPU確認
import torch

print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("GPUなし。LLM 4bitロードは失敗する可能性があります。mock実行は可能です。")


## 2. シミュレーションを実行する

このセルで複数サイクルの授業を回します。


In [ ]:
from pprint import pprint

from src.experiment import export_simulation_timeline_results, run_simulation_timeline

simulation_result = run_simulation_timeline(
    class_id=CLASS_ID,
    class_size=CLASS_SIZE,
    cycles=CYCLES,
    total_minutes=TOTAL_MINUTES,
    use_llm_student=USE_LLM_STUDENT,
    use_llm_communication=USE_LLM_COMMUNICATION,
    model_id=MODEL_ID,
    load_in_4bit=LOAD_IN_4BIT,
    cognitive_model_type=COGNITIVE_MODEL,
    update_student_knowledge=UPDATE_STUDENT_KNOWLEDGE,
)

simulation_outputs = export_simulation_timeline_results(simulation_result)

print("outputs:")
pprint(simulation_outputs)
print()
print("research_metrics:")
pprint(simulation_result["research_metrics"])
print()
print("issue_candidates:", len(simulation_result["issue_candidates"]))


## 3. 時間経過ごとの評価指標

サイクルごとに、研究発表で使えそうな指標を見ます。

- 正答率
- 認知モデル名
- 教師beliefの平均確信度
- 推定理解度
- 誤概念候補数
- 反応時間
- 授業目標と授業ペース


In [ ]:
import pandas as pd

cycle_df = pd.DataFrame(simulation_result["cycle_summary_rows"])
display(cycle_df)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(cycle_df["cycle_index"], cycle_df["accuracy"], marker="o")
axes[0].set_title("Accuracy over cycles")
axes[0].set_ylim(0, 1.05)

axes[1].plot(cycle_df["cycle_index"], cycle_df["average_belief_confidence"], marker="o", color="tab:orange")
axes[1].set_title("Teacher belief confidence")
axes[1].set_ylim(0, 1.05)

axes[2].plot(cycle_df["cycle_index"], cycle_df["average_estimated_score"], marker="o", color="tab:green")
axes[2].set_title("Estimated class understanding")
axes[2].set_ylim(0, 100)

for ax in axes:
    ax.set_xlabel("cycle")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. フェーズごとの時間経過

授業の各フェーズで、どこに問題が出るかを見ます。


In [ ]:
timeline_df = pd.DataFrame(simulation_result["timeline_rows"])
display(timeline_df[[
    "cycle_index", "phase", "accuracy", "priority_student_count",
    "low_self_efficacy_count", "low_question_tendency_count",
    "low_motivation_count", "high_neuroticism_count", "average_response_time_sec"
]])


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_df = timeline_df.copy()
plot_df["step"] = plot_df["cycle_index"].astype(str) + "-" + plot_df["phase"].astype(str)
ax.plot(plot_df["step"], plot_df["priority_student_count"], marker="o", label="priority students")
ax.plot(plot_df["step"], plot_df["high_neuroticism_count"], marker="o", label="high anxiety")
ax.plot(plot_df["step"], plot_df["low_question_tendency_count"], marker="o", label="low question tendency")
ax.tick_params(axis="x", rotation=45)
ax.set_title("Observed classroom risks over lesson phases")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 5. 問題候補を確認する

シミュレーションを回しながら、改善すべき箇所を抽出します。


In [ ]:
from pprint import pprint

pprint(simulation_result["issue_candidates"][:10])


## 6. 小規模LLM実行

Colab GPUで、まず3人・1サイクルだけLLMを入れて確認します。

ここでは、認知モデルが正誤方針を決め、LLMは次を担当します。

- 生徒AIの自然な発話生成
- 伝達AIによる発話分類・クラス要約

重い場合は `USE_LLM_STUDENT_SMALL = False` にして、伝達AIだけLLMにしてください。


In [ ]:
from pprint import pprint

USE_LLM_STUDENT_SMALL = True
USE_LLM_COMMUNICATION_SMALL = True
LLM_CLASS_SIZE = 3
LLM_CYCLES = 1

if not torch.cuda.is_available():
    raise RuntimeError("LLM実行にはColabのGPUランタイムを使ってください。mock実験はGPUなしでも実行できます。")

llm_simulation_result = run_simulation_timeline(
    class_id=CLASS_ID,
    class_size=LLM_CLASS_SIZE,
    cycles=LLM_CYCLES,
    total_minutes=TOTAL_MINUTES,
    use_llm_student=USE_LLM_STUDENT_SMALL,
    use_llm_communication=USE_LLM_COMMUNICATION_SMALL,
    model_id=MODEL_ID,
    load_in_4bit=LOAD_IN_4BIT,
    cognitive_model_type=COGNITIVE_MODEL,
    update_student_knowledge=UPDATE_STUDENT_KNOWLEDGE,
    teacher_beliefs_dir="data/teacher_beliefs/simulation_timeline_llm",
    logs_dir="data/logs/simulation_timeline_llm",
)

llm_simulation_outputs = export_simulation_timeline_results(
    llm_simulation_result,
    stem="simulation_timeline_llm",
)

print("LLM outputs:")
pprint(llm_simulation_outputs)
print("LLM conditions:")
pprint(llm_simulation_result["conditions"])
print("LLM research_metrics:")
pprint(llm_simulation_result["research_metrics"])
print("LLM issue_candidates:", len(llm_simulation_result["issue_candidates"]))


In [ ]:
llm_cycle = llm_simulation_result["cycles"][0]

print("LLM first student utterances:")
for event in llm_cycle["session_result"]["turns"][0]["events"]:
    print("---", event["student_id"])
    print(event["utterance"])

print()
print("LLM first classroom observation:")
pprint(llm_cycle["session_result"]["turns"][0]["classroom_observation"])

print()
print("LLM result txt preview:")
llm_summary_path = Path(llm_simulation_outputs["txt"])
print(llm_summary_path)
print(llm_summary_path.read_text(encoding="utf-8")[:3000])


## 7. 1サイクル分の中身を見る

LLMを入れたときは、ここで実際の発話・伝達AI要約・教師belief更新を確認します。


In [ ]:
cycle = simulation_result["cycles"][0]
print("cycle_metrics:")
pprint(cycle["cycle_metrics"])
print()
print("first turn classroom observation:")
pprint(cycle["session_result"]["turns"][0]["classroom_observation"])
print()
print("first event:")
pprint(cycle["session_result"]["turns"][0]["events"][0])


## 8. 共有用txtを表示する

結果をCodex/ChatGPTに渡す場合は、このtxtを添付してください。


In [ ]:
from pathlib import Path

summary_path = Path(simulation_outputs["txt"])
print(summary_path)
print(summary_path.read_text(encoding="utf-8")[:5000])


## 9. この環境で評価できる研究指標

発表・論文で使えそうな指標:

- cycleごとの正答率変化
- cycleごとのTeacher Belief確信度変化
- cycleごとの推定クラス理解度変化
- 授業目標・授業ペースの変化
- phaseごとの要支援生徒数
- phaseごとの不安・質問傾向・モチベーション推定
- 反応時間の変化
- issue_candidatesとして抽出される失敗例

今後追加したい指標:

- Oracle Plannerとの差
- LLM伝達AIとルールベース伝達AIの授業計画差
- 授業介入ON/OFFによる生徒状態変化
- 誤概念が減ったかどうかの推移
